In [1]:
from pathlib import Path

import gc
import json
import random

import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dropout,
    BatchNormalization,
    Dense,
)
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
)


SEED = 12345

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 35

FROZEN_EPOCHS = 200
FINETUNE_EPOCHS = 400

FROZEN_LEARNING_RATE = 1e-4
FINETUNE_LEARNING_RATE = 1e-7


random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Expected classes:", NUM_CLASSES)

TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Image size: (128, 128)
Batch size: 32
Expected classes: 35


In [2]:
KAGGLE_INPUT = Path("/kaggle/input")

MANIFEST_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_clean_split_manifest.csv")
)

CLASS_NAMES_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_class_names.json")
)

DATASET_ROOT = next(
    KAGGLE_INPUT.rglob("zooplankton_0p5x")
)


manifest_df = pd.read_csv(MANIFEST_PATH)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as file:
    CLASS_NAMES = json.load(file)


manifest_df["filepath"] = manifest_df["relative_filepath"].apply(
    lambda path: str(DATASET_ROOT / path)
)


train_df = manifest_df[
    manifest_df["split"] == "train"
].copy()

validation_df = manifest_df[
    manifest_df["split"] == "validation"
].copy()

test_df = manifest_df[
    manifest_df["split"] == "test"
].copy()


print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Classes:", len(CLASS_NAMES))

Training: 12558
Validation: 2691
Test: 2691
Classes: 35


In [3]:
train_datagen = ImageDataGenerator(
    rotation_range=180,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.20,
    shear_range=10,
)

eval_datagen = ImageDataGenerator()


train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
    interpolation="lanczos"
)

validation_generator = eval_datagen.flow_from_dataframe(
    dataframe=validation_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    interpolation="lanczos"
)

test_generator = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
    interpolation="lanczos"
)

Found 12558 validated image filenames belonging to 35 classes.
Found 2691 validated image filenames belonging to 35 classes.
Found 2691 validated image filenames belonging to 35 classes.


In [4]:
backbone = EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3),
)

backbone.trainable = False


x = GlobalAveragePooling2D()(backbone.output)
x = Dropout(0.1)(x)
x = BatchNormalization()(x)

x = Dense(
    450,
    activation="relu",
)(x)

x = Dropout(0.7)(x)
x = BatchNormalization()(x)

outputs = Dense(
    NUM_CLASSES,
    activation="softmax",
    kernel_initializer="random_uniform",
)(x)


model = Model(
    inputs=backbone.input,
    outputs=outputs,
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FROZEN_LEARNING_RATE,
    ),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy",
        ),
    ],
)


model.summary()

I0000 00:00:1786526399.218665      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 128, 128,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 128, 128,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 128, 128,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 129, 129,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 64, 64,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 64, 64,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 64, 64,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 64, 64,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 64, 64,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 64, 64,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 64, 64,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 64, 64,    │        512 │ block1a_se_excit

 Total params: 8,425,836 (32.14 MB)

 Trainable params: 653,551 (2.49 MB)

 Non-trainable params: 7,772,285 (29.65 MB)

In [5]:
FROZEN_PATIENCE = 50

OUTPUT_DIR = Path(
    "/kaggle/working/efficientnetb2_verified_head_frozen"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


BEST_WEIGHTS_PATH = (
    OUTPUT_DIR / "best_frozen.weights.h5"
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_frozen_model.keras"
)

HISTORY_PATH = (
    OUTPUT_DIR / "frozen_history.csv"
)


callbacks = [
    ModelCheckpoint(
        filepath=str(BEST_WEIGHTS_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),

    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=FROZEN_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.CSVLogger(
        str(HISTORY_PATH)
    ),
]

In [6]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=FROZEN_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

Epoch 1/200


I0000 00:00:1786526442.572042      67 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_loss improved from None to 1.64806, saving model to /kaggle/working/efficientnetb2_verified_head_frozen/best_frozen.weights.h5

Epoch 1: finished saving model to /kaggle/working/efficientnetb2_verified_head_frozen/best_frozen.weights.h5
393/393 - 244s - 622ms/step - accuracy: 0.3743 - loss: 2.6041 - top2_accuracy: 0.4857 - val_accuracy: 0.6938 - val_loss: 1.6481 - val_top2_accuracy: 0.8112
Epoch 2/200

Epoch 2: val_loss improved from 1.64806 to 1.06758, saving model to /kaggle/working/efficientnetb2_verified_head_frozen/best_frozen.weights.h5

Epoch 2: finished saving model to /kaggle/working/efficientnetb2_verified_head_frozen/best_frozen.weights.h5
393/393 - 81s - 205ms/step - accuracy: 0.6093 - loss: 1.6630 - top2_accuracy: 0.7467 - val_accuracy: 0.7544 - val_loss: 1.0676 - val_top2_accuracy: 0.8473
Epoch 3/200

Epoch 3: val_loss improved from 1.06758 to 0.87142, saving model to /kaggle/working/efficientnetb2_verified_head_frozen/best_frozen.weights.h5

Epoch 3: finish

In [7]:
model.load_weights(BEST_WEIGHTS_PATH)

model.save(BEST_MODEL_PATH)

print("Best weights saved at:")
print(BEST_WEIGHTS_PATH)

print("\nComplete model saved at:")
print(BEST_MODEL_PATH)

Best weights saved at:
/kaggle/working/efficientnetb2_verified_head_frozen/best_frozen.weights.h5

Complete model saved at:
/kaggle/working/efficientnetb2_verified_head_frozen/best_frozen_model.keras


In [8]:
from sklearn.metrics import f1_score


validation_generator.reset()

validation_results = model.evaluate(
    validation_generator,
    return_dict=True,
    verbose=1,
)


validation_generator.reset()

validation_probabilities = model.predict(
    validation_generator,
    verbose=1,
)

validation_predictions = np.argmax(
    validation_probabilities,
    axis=1,
)

validation_macro_f1 = f1_score(
    validation_generator.classes,
    validation_predictions,
    average="macro",
)


print("\nFrozen validation results:")

print(
    f"Loss: {validation_results['loss']:.4f}"
)

print(
    f"Accuracy: {validation_results['accuracy']:.4f}"
)

print(
    f"Top-2 accuracy: "
    f"{validation_results['top2_accuracy']:.4f}"
)

print(
    f"Macro-F1: {validation_macro_f1:.4f}"
)

85/85 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.9186 - loss: 0.2893 - top2_accuracy: 0.9602
85/85 ━━━━━━━━━━━━━━━━━━━━ 23s 164ms/step

Frozen validation results:
Loss: 0.2893
Accuracy: 0.9186
Top-2 accuracy: 0.9602
Macro-F1: 0.7648
